In [ ]:
!pip install -q torch_optimizer seedbank mlflow optuna pyngrok scipy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset, Dataset
import torch_optimizer as optim_extra
from torch_optimizer import Lamb
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from typing import Any, Optional
from google.colab.output import eval_js
from tabulate import tabulate
from PIL import Image
from numpy.typing import NDArray
from tqdm import tqdm

import time
import shutil
import random
import copy
import numpy as np
import mlflow
import mlflow.pytorch
import mlflow.onnx
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import yaml
from google.colab import drive
from google.colab import userdata
from pyngrok import ngrok
import os
import optuna
import pandas as pd
import scipy.stats as st

In [ ]:
import seedbank

seedbank.initialize(42)

## Подключение к mlflow и хранилищу


In [ ]:
drive.mount('/content/drive')

mlflow.set_tracking_uri("file:/content/drive/MyDrive/mlflow")

In [ ]:
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

get_ipython().system_raw("mlflow ui  --host 0.0.0.0 --backend-store-uri file:/content/drive/MyDrive/mlflow --port 5001 &")

public_url = ngrok.connect(5001)
print(" * MLflow UI доступен по адресу:", public_url)

## Загрузка датасета

In [ ]:
!mkdir -p ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {"username":"","key":	""}
import json
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(api_token, f)

!chmod 600 ~/.kaggle/kaggle.json
!cat ~/.kaggle/kaggle.json

!kaggle datasets download -d alessiocorrado99/animals10
!unzip -q animals10.zip

## Модель

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(16), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 16 * 16, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device, scheduler=None):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        running_loss += loss.item()

    return running_loss / len(train_loader)

In [ ]:
def evaluate_model(model, test_loader, device):
    all_labels = []
    all_predictions = []

    model.eval()

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions) * 100
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0) * 100
    recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0) * 100
    f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0) * 100

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

## Шумы

In [ ]:
class GaussianNoiseAdder(object):
    def __init__(self, mean=0., std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        x = tensor + torch.randn_like(tensor) * self.std + self.mean
        return x.clamp(0.0, 1.0)

    def __repr__(self):
        return f"{self.__class__.__name__}(mean={self.mean}, std={self.std})"


class SaltAndPepperNoiseAdder(object):
    def __init__(self, amount=0.04):
        self.amount = amount

    def __call__(self, tensor):
        noisy = tensor.clone()
        c, h, w = noisy.size()
        num = int(self.amount * h * w)

        if num == 0:
          return noisy

        idx = torch.randperm(h * w)[:num]
        r, col = idx // w, idx % w
        half = num // 2
        noisy[:, r[:half], col[:half]] = 1.0
        noisy[:, r[half:], col[half:]] = 0.0
        return noisy.clamp(0.0, 1.0)

    def __repr__(self):
        return f"{self.__class__.__name__}(amount={self.amount})"

## Генерация данных на гугл диске

In [ ]:
def generate_datasets_on_drive(config_path: str, noise_registry: dict):
    """
    Scans Google Drive and creates only noisy datasets that don't yet exist.
    Generates locally first (fast), then uploads only missing scenarios.
    """
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    source_path = config['data']['clean_data_path']
    root_path = config['data']['preprocessed_root_path']
    folder_template = config['data']['scenario_folder_template']

    print(f"Loading clean dataset from: {source_path}")
    clean_data = torchvision.datasets.ImageFolder(root=source_path)

    os.makedirs(root_path, exist_ok=True)
    print(f"Check and generate datasets in Drive path: {root_path}")

    for scenario_name, noise_config in config['grid_search']['noise_scenarios'].items():
        scenario_folder_name = folder_template.format(scenario_name=scenario_name)
        target_path = os.path.join(root_path, scenario_folder_name)

        if os.path.exists(target_path):
            print(f"Dataset for '{scenario_name}' already exists on Drive. Skip.")
            continue

        local_temp = f"/content/tmp_{scenario_folder_name}"
        if os.path.exists(local_temp):
            shutil.rmtree(local_temp)
        os.makedirs(local_temp, exist_ok=True)

        print(f"\nGenerating dataset '{scenario_name}' locally in '{local_temp}'...")

        noise_transforms = [noise_registry[n['name']](**n['params']) for n in noise_config]

        transform_pipeline = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            *noise_transforms,
            transforms.ToPILImage()
        ])

        for img_path, label_idx in tqdm(clean_data.imgs, desc=f"  Scenario {scenario_name}"):
            try:
                img = Image.open(img_path).convert("RGB")
                processed_img = transform_pipeline(img)

                class_name = clean_data.classes[label_idx]
                local_class_path = os.path.join(local_temp, class_name)
                os.makedirs(local_class_path, exist_ok=True)

                img_name = os.path.basename(img_path)
                processed_img.save(os.path.join(local_class_path, img_name))

            except Exception as e:
                print(f"Failed to process file {img_path}: {e}")

        print(f"Uploading scenario '{scenario_name}' to Google Drive...")
        shutil.copytree(local_temp, target_path)
        print(f"Uploaded: {target_path}")

        sync_seconds = 900
        print(f"Waiting {sync_seconds//60} minutes for Google Drive to sync...")

        for _ in tqdm(range(sync_seconds), desc="Google Drive syncing"):
            time.sleep(1)

        print("Synchronization time completed. Proceeding to next scenario.")

        shutil.rmtree(local_temp)

    print("\nDataset verification and generation are completed.")

In [ ]:
NOISE_REGISTRY = {
    "GaussianNoiseAdder": GaussianNoiseAdder,
    "SaltAndPepperNoiseAdder": SaltAndPepperNoiseAdder,
}

generate_datasets_on_drive(config_path='test_config.yaml', noise_registry=NOISE_REGISTRY)

## Utils

In [ ]:
def calculate_aggregated_metrics(run_results_list: list[dict[str, Any]]) -> dict[str, Any]:
    """
    Calculates aggregated metrics over a list of results.

    Returns:
        A dictionary with the full structure of calculated metrics. Example:
        {
            'runs_count': 3,
            'validation_metrics': { ... },
            'test_metrics': {
                'accuracy': {
                    'mean': 92.33,
                    'ci_95': (91.4, 93.2),
                    'formatted': "92.33 (91.40, 93.20)"
                },
                'f1_score': { ... }
            },
            'mean_time_s': 111.23
        }
    """
    if not run_results_list:
        return {}

    num_runs = len(run_results_list)
    aggregated_results = {
        "runs_count": num_runs,
        "validation_metrics": {},
        "test_metrics": {},
        "mean_time_s": None
    }

    final_val_metrics = [run['val_metrics_history'][-1] for run in run_results_list if run.get('val_metrics_history')]
    if final_val_metrics:
        val_metric_keys = final_val_metrics[0].keys()
        aggregated_val = {key: [d.get(key) for d in final_val_metrics if d.get(key) is not None] for key in val_metric_keys}
        for name, values in aggregated_val.items():
            if values:
                mean_val = np.mean(values)
                std_val = np.std(values) if num_runs > 1 else 0.0

                aggregated_results["validation_metrics"][name] = {
                    'mean': mean_val,
                    'std': std_val
                }

    test_metrics = [run.get('metrics') for run in run_results_list if run.get('metrics')]
    if test_metrics:
        test_metric_keys = test_metrics[0].keys()
        aggregated_test = {key: [d.get(key) for d in test_metrics if d.get(key) is not None] for key in test_metric_keys}
        for name, values in aggregated_test.items():
            if not values:
                continue

            mean_val = np.mean(values)
            std_val = np.std(values) if num_runs > 1 else 0.0
            ci_95 = (mean_val, mean_val)
            if num_runs > 1:
                sem = st.sem(values)
                if sem > 0:
                    ci_95 = st.t.interval(confidence=0.95, df=num_runs - 1, loc=mean_val, scale=sem)

            aggregated_results["test_metrics"][name] = {
                'mean': mean_val,
                'ci_95_lower': max(0, ci_95[0]),
                'ci_95_upper': ci_95[1],
                'std': std_val,
                'formatted': f"{mean_val:.2f} ({max(0, ci_95[0]):.2f}, {ci_95[1]:.2f})"
            }

    times = [run.get('time_metric') for run in run_results_list if run.get('time_metric') is not None]
    if times:
        aggregated_results["mean_time_s"] = np.mean(times)
        aggregated_results["time_std_s"] = np.std(times) if num_runs > 1 else 0.0

    return aggregated_results

def generate_summary_table(data: list[dict]) -> None:
    """Prints the resulting pivot table to the console."""
    print("\n\n" + "="*100)
    print(" " * 40 + "FINAL SUMMARY TABLE")
    print("="*100)

    if not data:
        print("There is no data to display in the summary.")
        return

    summary_df = pd.DataFrame(data)
    table_str = tabulate(summary_df, headers='keys', tablefmt='grid',
                        showindex=False, numalign='center', stralign='center')

    print(table_str)

def save_summary_to_csv(summary_data: list[dict], filename: str = "experiment_summary.csv") -> None:
    """
    Saves a complete summary of experiments to a CSV file.
    """
    if not summary_data:
        print("There is no data to save to CSV.")
        return

    records = []
    for row in summary_data:
        record = {
            "experiment": row["experiment"],
            "hyperparams": row["hyperparams"],
            "epochs_num": row["epochs_num"],
            "conv_time_mean_s": row["mean_time_s"],
            "conv_time_std_s": row["time_std_s"],
        }

        for metric_name, data in row['full_metrics'].items():
            record[f'{metric_name}_mean'] = data.get('mean')
            record[f'{metric_name}_std'] = data.get('std')
            record[f'{metric_name}_ci95_lower'] = data.get('ci_95_lower')
            record[f'{metric_name}_ci95_upper'] = data.get('ci_95_upper')

        records.append(record)

    try:
        df = pd.DataFrame(records)
        df.to_csv(filename, index=False, float_format='%.2f')
        print(f"\nThe full summary was successfully saved to file: {filename}")
    except Exception as e:
        print(f"\nError saving CSV file: {e}")


In [ ]:
def get_dataloaders_from_drive(
    preprocessed_root_path: str,
    scenario_folder_template: str,
    scenario_name: str,
    random_state: int,
    batch_size: int,
    subset_size: int = None
    ) -> tuple[DataLoader, DataLoader, DataLoader]:
    """
    Loads a preprocessed dataset from Google Drive based on a scenario name,
    splits it into training, validation, and test sets, and returns their respective DataLoaders.

    Args:
        preprocessed_root_path (str): The path to the root directory where all
                                      preprocessed datasets are stored.
        scenario_folder_template (str): A format string for the scenario subfolder name,
                                        e.g., 'Animals10_{scenario_name}'.
        scenario_name (str): The specific name of the scenario to load,
                             e.g., 'gaussian_0.03'.
        random_state (int): The seed for the random number generator to ensure
                            reproducible train/val/test splits.
        batch_size (int): The number of samples per batch to load.
        subset_size (int, optional): If specified, a random subset of this size
                                     is used instead of the full dataset.
                                     Defaults to None (using the full dataset).

    Returns:
        A tuple containing three DataLoader objects: (train_loader, val_loader, test_loader).

    Raises:
        FileNotFoundError: If the directory for the specified scenario does not exist.
    """
    scenario_folder_name = scenario_folder_template.format(scenario_name=scenario_name)
    data_path = os.path.join(preprocessed_root_path, scenario_folder_name)

    if not os.path.exists(data_path):
        error_msg = (
            f"Directory for scenario '{scenario_name}' not found at the expected path: {data_path}\n"
            f"Please ensure you have run the preprocessing script to generate this dataset."
        )
        raise FileNotFoundError(error_msg)

    print(f"    - Loading data from: {data_path}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    full_dataset = torchvision.datasets.ImageFolder(root=data_path, transform=transform)

    if subset_size:
        print(f"    - DEBUG MODE:  Running on a slice of {subset_size} / {len(full_dataset)} samples\n")
        subset_size = min(subset_size, len(full_dataset))
        indices = np.arange(subset_size)
        full_dataset = Subset(full_dataset, indices)

    train_val_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_val_size
    train_val_data, test_data = random_split(
        full_dataset, [train_val_size, test_size],
        generator=torch.Generator().manual_seed(random_state)
    )

    train_size = int(0.85 * len(train_val_data))
    val_size = len(train_val_data) - train_size
    train_data, val_data = random_split(
        train_val_data, [train_size, val_size],
        generator=torch.Generator().manual_seed(random_state)
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [ ]:
def set_random_seed(seed: int):
    """
    Sets the seed for all major random number generators to ensure reproducibility.

    Args:
        seed (int): The integer value to use as the seed.
    """
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

## Пример конфига

In [ ]:
# mlflow:
#   experiment_name: "{model_name}_{dataset_name}"
#   # tags:
#   #   project: "Noisy Optimizers"

# data:
#   dataset_name: "Animals10"
#   clean_data_path: "raw-img"
#   preprocessed_root_path: "/content/drive/MyDrive/ml_data/Animals10_Datasets"
#   scenario_folder_template: "Animals10_{scenario_name}"
#   num_classes: 10
#   debug_subset_size: 100

# model:
#   name: "SimpleCNN"
#   params:
#     num_classes: 10

# training:
#   epochs: 12
#   batch_size: 64
#   learning_rate: 0.001
#   target_loss: 0.4
#   criterion: "CrossEntropyLoss"
#   num_runs: 3
#   save_model_mode: "none" #best, all

# grid_search:
#   optimizers:
# #    - name: "Adam"
# #      params: {}
#     - name: "SGD"
#       params:
#         momentum: 0.9

#   noise_scenarios:
#     no_noise: []
#     gaussian_0.03:
#       - name: "GaussianNoiseAdder"
#         params:
#           mean: 0.0
#           std: 0.03
#     gaussian_0.05:
#       - name: "GaussianNoiseAdder"
#         params:
#           mean: 0.0
#           std: 0.05
# #    gaussian_0.1:
# #      - name: "GaussianNoiseAdder"
# #        params:
# #          mean: 0.0
# #          std: 0.1
#     gaussian_0.13:
#       - name: "GaussianNoiseAdder"
#         params:
#           mean: 0.0
#           std: 0.13
# #    gaussian_0.17:
# #      - name: "GaussianNoiseAdder"
# #        params:
# #          mean: 0.0
# #          std: 0.17
# #    gaussian_0.20:
# #      - name: "GaussianNoiseAdder"
# #        params:
# #          mean: 0.0
# #          std: 0.20

#     salt_pepper_0.02:
#       - name: "SaltAndPepperNoiseAdder"
#         params:
#           amount: 0.02
#     salt_pepper_0.03:
#       - name: "SaltAndPepperNoiseAdder"
#         params:
#           amount: 0.03
#     salt_pepper_0.07:
#       - name: "SaltAndPepperNoiseAdder"
#         params:
#           amount: 0.07
# #    salt_pepper_0.09:
# #      - name: "SaltAndPepperNoiseAdder"
# #        params:
# #          amount: 0.09


## Основной эксперимент

In [ ]:
class ExperimentRunner:
    """
    A class for managing, running, and logging experiments based on a given configuration.
    """
    def __init__(self, config_path: str, model_registry: dict[str, nn.Module],
                 noise_registry: dict[str, nn.Module], optimizer_registry: dict[str, nn.Module]) -> None:
        """
        Initializes ExperimentRunner.

        Args:
            config_path: Path to the YAML configuration file.
            model_registry: Dictionary of available model classes.
            noise_registry: Dictionary with available noise/transformation classes.
            optimizer_registry: Dictionary of available optimizer classes.
        """
        self.config_path = config_path
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)

        self.model_registry = model_registry
        self.noise_registry = noise_registry
        self.optimizer_registry = optimizer_registry
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self._setup_mlflow()

    def _setup_mlflow(self) -> None:
        """Sets up an experiment in MLflow."""
        exp_name_template = self.config['mlflow']['experiment_name']
        model_name = self.config['model']['name']
        dataset_name = self.config['data']['dataset_name']
        experiment_name = exp_name_template.format(model_name=model_name, dataset_name=dataset_name)
        mlflow.set_experiment(experiment_name)
        print(f"MLflow experiment set to: '{experiment_name}'")

    def _get_model(self) -> nn.Module:
        """Creates and returns an instance of the model according to the config."""
        model_name = self.config['model']['name']
        model_class = self.model_registry[model_name]
        model_params = self.config['model'].get('params', {})
        return model_class(**model_params).to(self.device)

    def _get_criterion(self) -> nn.Module:
        """Creates and returns a loss function."""
        criterion_name = self.config['training']['criterion']
        return getattr(nn, criterion_name)()

    def _run_single_experiment(self, base_run_name: str, scenario_name: str, opt_config: dict, criterion: nn.Module, run_seeds: NDArray) -> list[dict[str, Any]]:
        """
        Performs one full experiment (opt + scenario) with N runs.
        """
        run_results_list = []
        training_params = self.config['training']
        save_mode = training_params.get('save_model_mode', 'best')
        num_runs = training_params.get('num_runs', 1)
        opt_name = opt_config['name']
        target_loss = training_params.get('target_loss', 0.3)

        print(f"\n--- Running {base_run_name} ({num_runs} times) ---")

        with mlflow.start_run(run_name=base_run_name) as parent_run:
            mlflow.log_params(self.config['model']['params'])
            mlflow.log_params(opt_config['params'])
            mlflow.log_param("optimizer_name", opt_name)
            mlflow.log_param("num_runs", num_runs)

            best_overall_run_accuracy = -1.0
            best_overall_model_state = None
            best_run_index = -1

            train_loader, val_loader, test_loader = get_dataloaders_from_drive(
              preprocessed_root_path=self.config['data']['preprocessed_root_path'],
              scenario_folder_template=self.config['data']['scenario_folder_template'],
              scenario_name=scenario_name,
              random_state=42,
              batch_size=self.config['training']['batch_size'],
              subset_size=self.config['data'].get('debug_subset_size')
              )

            input_example = next(iter(train_loader))[0][:1].cpu().numpy()
            signature = infer_signature(input_example)

            for i, run_seed in enumerate(run_seeds):
                with mlflow.start_run(run_name=f"{base_run_name}_run_{i+1}", nested=True):
                    mlflow.log_param("random_state", run_seed)
                    set_random_seed(int(run_seed))

                    model = self._get_model()
                    optimizer_class = self.optimizer_registry[opt_name]
                    optimizer = optimizer_class(model.parameters(), lr=training_params['learning_rate'], **opt_config['params'])

                    val_metrics_history = []
                    convergence_time = -1
                    start_time = time.time()

                    for epoch in range(training_params['epochs']):
                        epoch_loss = train_one_epoch(model, optimizer, criterion, train_loader, self.device)
                        val_metrics = evaluate_model(model, val_loader, self.device)
                        val_metrics_history.append(val_metrics)

                        if epoch_loss <= target_loss and convergence_time < 0:
                            convergence_time = time.time() - start_time

                        mlflow.log_metric("epoch_loss", epoch_loss, step=epoch)
                        for name, value in val_metrics.items():
                            mlflow.log_metric(f"val_{name}", value, step=epoch)

                    total_time = time.time() - start_time

                    time_to_log = convergence_time if convergence_time > 0 else total_time
                    mlflow.log_metric("convergence_time", time_to_log)

                    test_metrics = evaluate_model(model, test_loader, self.device)

                    run_results_list.append({
                        "metrics": test_metrics,
                        "time_metric": time_to_log,
                        "val_metrics_history": val_metrics_history
                    })

                    mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})

                    final_run_val_accuracy = val_metrics_history[-1].get('accuracy', 0) if val_metrics_history else 0
                    mlflow.log_metric("final_val_accuracy", final_run_val_accuracy)

                    if save_mode == "all":
                        print(f"    - Saving the model (Val Acc: {final_run_val_accuracy:.2f}%)")
                        mlflow.pytorch.log_model(model, name="final_model",signature=signature )

                    if save_mode == "best":
                        if final_run_val_accuracy > best_overall_run_accuracy:
                            print(f"    - A new best launch has been found (Val Acc: {final_run_val_accuracy:.2f}%)")
                            best_overall_run_accuracy = final_run_val_accuracy
                            best_run_index = i + 1
                            best_overall_model_state = copy.deepcopy(model.state_dict())

            if save_mode == "best" and best_overall_model_state is not None:
                print(f"\nSaving the BEST model from {num_runs} runs (from run #{best_run_index} with Val Acc: {best_overall_run_accuracy:.2f}%)")
                best_model = self._get_model()
                best_model.load_state_dict(best_overall_model_state)
                mlflow.pytorch.log_model(best_model, name="best_model_across_runs", signature=signature)
                mlflow.log_metric("best_run_val_accuracy", best_overall_run_accuracy)
                mlflow.log_param("best_run_index", best_run_index)

            if not run_results_list:
                print("Warning: No runs were performed, aggregation is not possible.")
                return {}

            aggregated_metrics = calculate_aggregated_metrics(run_results_list)

            print(f"\n--- Aggregation and logging for '{base_run_name}' ---")
            if aggregated_metrics:
                mean_time = aggregated_metrics.get('mean_time_s', 0)
                std_time = aggregated_metrics.get('time_std_s', 0)
                mlflow.log_metric("mean_time", mean_time)
                mlflow.log_metric("std_time", std_time)

                for name, data in aggregated_metrics.get('test_metrics', {}).items():
                    mlflow.log_metric(f"test_{name}_mean", data['mean'])
                    mlflow.log_metric(f"test_{name}_std", data['std'])
                    mlflow.log_metric(f"test_{name}_ci95_lower", data['ci_95_lower'])
                    mlflow.log_metric(f"test_{name}_ci95_upper", data['ci_95_upper'])

                for name, data in aggregated_metrics.get('validation_metrics', {}).items():
                    mlflow.log_metric(f"val_{name}_mean", data['mean'])
                    mlflow.log_metric(f"val_{name}_std", data['std'])

            if aggregated_metrics:
                mean_val_acc = aggregated_metrics.get('validation_metrics', {}).get('accuracy', {}).get('mean', 0)
                std_val_acc = aggregated_metrics.get('validation_metrics', {}).get('accuracy', {}).get('std', 0)
                mean_val_f1 = aggregated_metrics.get('validation_metrics', {}).get('f1_score', {}).get('mean', 0)
                std_val_f1 =  aggregated_metrics.get('validation_metrics', {}).get('f1_score', {}).get('std', 0)

                mean_test_acc = aggregated_metrics.get('test_metrics', {}).get('accuracy', {}).get('mean', 0)
                std_test_acc = aggregated_metrics.get('test_metrics', {}).get('accuracy', {}).get('std', 0)
                mean_test_f1 = aggregated_metrics.get('test_metrics', {}).get('f1_score', {}).get('mean', 0)
                std_test_f1 = aggregated_metrics.get('test_metrics', {}).get('f1_score', {}).get('std', 0)

                print(f"  - Val Accuracy: {mean_val_acc:.2f} ± {std_val_acc:.2f}")
                print(f"  - Val F1-score: {mean_val_f1:.2f} ± {std_val_f1:.2f}")
                print(f"  - Conv. Time, s: {mean_time:.2f} ± {std_time:.2f}")
                print(f"\n  - Test Accuracy: {mean_test_acc:.2f} ± {std_test_acc:.2f}")
                print(f"  - Test F1-score: {mean_test_f1:.2f} ± {std_test_f1:.2f}")

            return aggregated_metrics

    def run(self) -> None:
        """
        Launches the entire grid of experiments.
        """
        print(f"The experiment is running on the device: {self.device}")

        criterion = self._get_criterion()
        summary_data_full = []
        num_runs = self.config['training'].get('num_runs', 1)

        run_seeds = np.random.randint(0, 2**32 - 1, size=num_runs)
        print(f"\nGenerated seeds: {run_seeds}\n")

        for scenario_name, noise_config in self.config['grid_search']['noise_scenarios'].items():
            print(f"\n{'='*80}\nSCENARIO: {scenario_name}\n{'='*80}")

            for opt_config in self.config['grid_search']['optimizers']:
                base_run_name = f"{opt_config['name']}_{scenario_name}"

                run_results = self._run_single_experiment(base_run_name, scenario_name, opt_config, criterion, run_seeds)

                summary_row = {
                    "experiment": base_run_name,
                    "hyperparams": str(opt_config.get('params', 'default')),
                    "epochs_num": self.config['training']['epochs'],
                    "mean_time_s": run_results.get('mean_time_s', 0),
                    "time_std_s": run_results.get('time_std_s', 0),
                    "full_metrics": run_results.get('test_metrics', {})
                }
                summary_data_full.append(summary_row)

        summary_data_console = []
        for row in summary_data_full:
            console_row = {
                "Experiment": row["experiment"],
                "Conv. Time, s": f"{row['mean_time_s']:.2f} ± {row['time_std_s']:.2f}"
            }

            for name, data in row['full_metrics'].items():
                console_row[f"{name.capitalize()}, %"] = f"{data.get('mean', 0):.2f} ± {data.get('std', 0):.2f}"
            summary_data_console.append(console_row)

        generate_summary_table(summary_data_console)
        save_summary_to_csv(summary_data_full)

        print("\nAll experiments were completed successfully.")



## Использование

In [ ]:
MODEL_REGISTRY = {
    "SimpleCNN": SimpleCNN
}

NOISE_REGISTRY = {
    "GaussianNoiseAdder": GaussianNoiseAdder,
    "SaltAndPepperNoiseAdder": SaltAndPepperNoiseAdder,
}

OPTIMIZER_REGISTRY = {
    #"Adam": optim.Adam,
    "SGD": optim.SGD,
    #"Lamb": torch_optim.Lamb,
}


In [ ]:
mlflow.end_run()

In [ ]:
config_path = 'test_config.yaml'

runner = ExperimentRunner(
    config_path=config_path,
    model_registry=MODEL_REGISTRY,
    noise_registry=NOISE_REGISTRY,
    optimizer_registry=OPTIMIZER_REGISTRY
)

runner.run()

## Подбор гиперпараметров

In [ ]:
class HyperparameterTuner:
    def __init__(self, model_class, model_params, optimizer_name: str, train_loader, val_loader,
                 criterion=nn.CrossEntropyLoss(), epochs_per_trial=12):
        """
        Initializes the tuner.

        Args:
            model_class: Model class (SimpleCNN).
            optimizer_name (str): Name of the optimizer for selection ('SGD', 'Adam' или 'LAMB').
            train_loader: DataLoader for train data.
            val_loader: DataLoader for validation data.
            criterion: Loss function.
            epochs_per_trial (int): Number of epochs for one trial.
        """
        self.model_class = model_class
        self.model_params = model_params
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.epochs_per_trial = epochs_per_trial

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"The tuner will use the device: {self.device}")

        self._suggestion_methods = {
            "SGD": self._suggest_sgd_params,
            "Adam": self._suggest_adam_params,
            "LAMB": self._suggest_lamb_params,
        }

        if optimizer_name not in self._suggestion_methods:
            raise ValueError(f"Optimizer '{optimizer_name}' does't support. "
                             f"Available: {list(self._suggestion_methods.keys())}")

        self.optimizer_name = optimizer_name
        print(f"The tuner is configured to select parameters {self.optimizer_name}")

    def _suggest_sgd_params(self, trial):
        """Search space for SGD."""
        return {
            "lr": 10 ** trial.suggest_float("lr_log10", -4, 0),
            "momentum": trial.suggest_float("momentum", 0.5, 0.99),
            "nesterov": trial.suggest_categorical("nesterov", [False, True]),
            "weight_decay": 10 ** trial.suggest_float("wd_log10", -6, -3.3),
            "warmup_frac": trial.suggest_float("warmup_frac", 0.0, 0.1)
        }

    def _suggest_adam_params(self, trial):
        """Search space for Adam."""
        return {
            "lr": 10 ** trial.suggest_float("lr_log10", -5, -2),
            "betas": (trial.suggest_float("adam_beta1", 0.8, 0.99), trial.suggest_float("adam_beta2", 0.9, 0.999)),
            "weight_decay": 10 ** trial.suggest_float("wd_log10", -6, -3),
        }

    def _suggest_lamb_params(self, trial):
        """Search space for LAMB."""
        return {
            "lr": 10 ** trial.suggest_float("lr_log10", -4, -1),
            "betas": (trial.suggest_float("lamb_beta1", 0.8, 0.99), trial.suggest_float("lamb_beta2", 0.9, 0.999)),
            "weight_decay": 10 ** trial.suggest_float("wd_log10", -5, -2),
        }

    def objective(self, trial):
        model = self.model_class(**self.model_params).to(self.device)

        suggestion_func = self._suggestion_methods[self.optimizer_name]
        params = suggestion_func(trial)

        scheduler = None

        if self.optimizer_name == "SGD":
            warmup_fraction = params.pop("warmup_frac", 0.0)
            optimizer = optim.SGD(model.parameters(), **params)

            if warmup_fraction > 0:
                scheduler = OneCycleLR(
                    optimizer,
                    max_lr=params['lr'],
                    epochs=self.epochs_per_trial,
                    steps_per_epoch=len(self.train_loader),
                    pct_start=warmup_fraction
                )

        elif self.optimizer_name == "Adam":
            optimizer = optim.Adam(model.parameters(), **params)
        elif self.optimizer_name == "LAMB":
            optimizer = Lamb(model.parameters(), **params)

        for epoch in range(self.epochs_per_trial):
            train_one_epoch(model, optimizer, self.criterion, self.train_loader, self.device, scheduler)

        validation_metrics = evaluate_model(model, self.val_loader, self.device)
        metric_to_optimize = validation_metrics['accuracy']

        return metric_to_optimize

    def tune(self, n_trials=100, direction="maximize", timeout=None):
        """
        Starts the hyperparameter selection process.

        Args:
            n_trials (int): Number of selection iterations.
            direction (str): Optimization direction ("maximize" or "minimize").
            timeout (int, optional): Maximum time for selection in seconds.
        """
        study = optuna.create_study(direction=direction)
        study.optimize(self.objective, n_trials=n_trials, timeout=timeout)

        print("\nThe selection process is completed.")
        print(f"Best Accuracy for {self.optimizer_name}: {study.best_value:.4f}")
        print("With hyperparams:")
        for key, value in study.best_params.items():
            print(f"  - {key}: {value}")

        return study

In [ ]:
config_path = 'test_config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

root_path = config['data']['preprocessed_root_path']
template = config['data']['scenario_folder_template']
scenario_to_load = 'no_noise'
batch_size = config['training']['batch_size']
subset_size = config['data'].get('debug_subset_size')

model_params = config['model'].get('params', {})

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders_from_drive(
    preprocessed_root_path=root_path,
    scenario_folder_template=template,
    scenario_name=scenario_to_load,
    random_state=42,
    batch_size=batch_size,
    subset_size=subset_size
)

### Подбор гиперпараметров для SGD


In [ ]:
sgd_tuner = HyperparameterTuner(
    model_class=SimpleCNN,
    model_params=model_params,
    optimizer_name='SGD',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs_per_trial=10
)

In [ ]:
sgd_study = sgd_tuner.tune(n_trials=40)

=====================================================================================================

## Сценарий: обучение на чистых данных - тестирование на разных зашумленных сценариях

In [ ]:
def find_best_model_uri(experiment_name: str,
                            optimizer_name: str,
                            scenario_name: str) -> Optional[str]:
    """
    Finds the URI of the best model by searching through the run history.

    Can work with:
    1. save_mode='best' (searches for the model in the Parent Run using the best_run_val_accuracy metric).
    2. save_mode='all' (searches for the best Child Run).
    3. Situations where the last run crashed or produced a poor result
    (the function will automatically search previous runs).
    """
    print(f"Searching for best model for '{optimizer_name}_{scenario_name}'...")

    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        print(f" Experiment '{experiment_name}' not found.")
        return None

    parent_run_name = f"{optimizer_name}_{scenario_name}"

    filter_string = f"tags.mlflow.runName = '{parent_run_name}'"
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=filter_string,
        order_by=["attribute.start_time DESC"]
    )

    if not runs:
        print(f" No runs found with name '{parent_run_name}'.")
        return None

    print(f"  - Found {len(runs)} historical runs. Checking for valid artifacts...")

    for run in runs:
        run_id = run.info.run_id
        metrics = run.data.metrics

        if "best_run_val_accuracy" in metrics:
            acc = metrics["best_run_val_accuracy"]
            print(f"  - Found valid Parent Run (ID: {run_id}) with save_mode='best'.")
            print(f"    Validation Accuracy: {acc:.2f}%")
            return f"runs:/{run_id}/best_model_across_runs"

        child_filter = f"tags.mlflow.parentRunId = '{run_id}'"
        child_runs = client.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string=child_filter,
            order_by=["metrics.final_val_accuracy DESC"],
            max_results=1
        )

        if child_runs:
            best_child = child_runs[0]
            child_acc = best_child.data.metrics.get('final_val_accuracy', 0)

            if child_acc > 0:
                print(f"  - Found valid Child Run (ID: {best_child.info.run_id}) in Parent ({run_id}).")
                print(f"    Validation Accuracy: {child_acc:.2f}%")
                return f"runs:/{best_child.info.run_id}/final_model"

    print(f" Could not find ANY valid model for '{parent_run_name}' in history.")
    return None


In [ ]:
def run_comparative_robustness_evaluation(config_path: str, optimizer_registry: dict[str, Any]):
    """
    Evaluates and compares the robustness of the best models using the new finder logic.
    """
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
    except FileNotFoundError:
        print(f"ERROR: Configuration file not found at '{config_path}'")
        return

    experiment_name = config['mlflow']['experiment_name'].format(
        model_name=config['model']['name'],
        dataset_name=config['data']['dataset_name']
    )

    OPTIMIZERS_TO_COMPARE = list(optimizer_registry.keys())
    print(f"Starting comparative robustness evaluation for optimizers: {OPTIMIZERS_TO_COMPARE}")

    SCENARIO_TRAINED_ON = list(config['grid_search']['noise_scenarios'].keys())[0]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    all_results: list[dict[str, Any]] = []

    for optimizer_name in OPTIMIZERS_TO_COMPARE:
        print("\n" + "="*80)
        print(f"PROCESSING OPTIMIZER: {optimizer_name}")
        print("="*80)

        model_uri = find_best_model_uri(
            experiment_name=experiment_name,
            optimizer_name=optimizer_name,
            scenario_name=SCENARIO_TRAINED_ON
        )

        if not model_uri:
            print(f" Skipping '{optimizer_name}' as no valid champion model was found.")
            continue

        print(f"Loading model from URI: {model_uri}...")
        try:
            model = mlflow.pytorch.load_model(model_uri, map_location=device)
            model.eval()
        except Exception as e:
            print(f" ERROR: Failed to load model for '{optimizer_name}'. Skipping. Error: {e}")
            continue

        scenarios_to_test = config['grid_search']['noise_scenarios'].keys()
        for scenario_name in tqdm(scenarios_to_test, desc=f"Evaluating {optimizer_name}"):
            try:
                _, _, test_loader = get_dataloaders_from_drive(
                    preprocessed_root_path=config['data']['preprocessed_root_path'],
                    scenario_folder_template=config['data']['scenario_folder_template'],
                    scenario_name=scenario_name,
                    random_state=42,
                    batch_size=config['training']['batch_size'],
                    subset_size=config['data'].get('debug_subset_size')
                )

                with torch.no_grad():
                    metrics = evaluate_model(model, test_loader, device)

                row = {"optimizer": optimizer_name, "test_scenario": scenario_name}
                row.update(metrics)
                all_results.append(row)
            except Exception as e:
                print(f" ERROR: Failed scenario '{scenario_name}' for '{optimizer_name}': {e}")

    if not all_results:
        print("\n No results were collected. Cannot generate a summary table.")
        return

    results_df = pd.DataFrame(all_results)
    METRIC_TO_SHOW = 'accuracy'

    try:
        pivot_df = results_df.pivot_table(
            index='test_scenario',
            columns='optimizer',
            values=METRIC_TO_SHOW
        )

        desired_order = list(config['grid_search']['noise_scenarios'].keys())
        existing_order = [s for s in desired_order if s in pivot_df.index]
        pivot_df = pivot_df.reindex(existing_order)

        print("\n\n" + "="*80)
        print(f" " * 15 + f"COMPARATIVE ROBUSTNESS SUMMARY (Metric: {METRIC_TO_SHOW})")
        print("="*80)

        pd.set_option('display.width', 1000)
        pd.set_option('display.max_columns', 20)
        pd.set_option('display.precision', 2)
        print(pivot_df)
        print("="*80)

    except Exception as e:
        print(f" Failed to create a pivot table: {e}. Displaying raw data:")
        print(results_df)

    csv_filename = "comparative_robustness_evaluation.csv"
    results_df.to_csv(csv_filename, index=False, float_format='%.4f')
    print(f"\n Full results saved to '{csv_filename}'")

In [ ]:
run_comparative_robustness_evaluation(
    config_path='test_config.yaml',
    optimizer_registry=OPTIMIZER_REGISTRY
)